# GCI Global 2026 — Home Credit Default Risk
## Pipeline completo y reproducible

**Objetivo:** estimar la probabilidad de que cada cliente caiga en mora (`TARGET = 1`).

**Métrica:** ROC-AUC. Mide qué tan bien el modelo *ordena* a los clientes de más a menos riesgo. No importa si la probabilidad es 0.3 o 0.03, sino que los morosos queden por encima de los que pagan.

### Componentes principales
| Componente | Qué aporta |
|---|---|
| Variables financieras y de `EXT_SOURCE` | Capacidad de pago, puntajes externos, tiempos relativos y faltantes |
| **Tasa de interés implícita** | La evaluación de riesgo del propio prestamista, despejada de crédito y cuota |
| **Clientes repetidos** | Reconstrucción de la identidad del cliente y mora de sus otras solicitudes |
| **Imputación de `EXT_SOURCE`** | Segunda vista de los datos con los puntajes faltantes estimados |
| **Ensamble** | LightGBM y CatBoost × 2 vistas, reentrenados con el 100% de los datos × 3 semillas |

### Estructura
1. Configuración
2. Carga y validación de datos
3. Variables: base, tasa implícita, clientes repetidos, imputación de EXT (vista B)
4. Validación cruzada
5. Paso 1: validación cruzada de LightGBM y CatBoost × vistas A/B
6. Paso 2: reentrenamiento con el 100% de los datos × 3 semillas
7. Ensamble
8. Predicción y archivo de envío

**Fuentes y créditos.** Las variables de razones financieras y combinaciones de `EXT_SOURCE` se inspiran en notebooks públicos de la competencia original de Kaggle *Home Credit Default Risk* (2018). La variable de tasa de interés implícita se basa en la fórmula estándar de amortización francesa. Este notebook se elaboró con ayuda de Claude (Anthropic).


## 1. Configuración

- `SEED`: la semilla fija todos los procesos aleatorios. El README la exige para que cualquiera pueda reproducir tu CSV.
- `USE_GPU`: en Colab, si activas *Entorno de ejecución → Cambiar tipo → GPU T4*, CatBoost se entrena mucho más rápido. En CPU, desde cero, el notebook tarda aproximadamente **1.5 horas**.
- `USE_CACHE`: guarda los resultados de cada modelo en Drive (carpeta `cache`). Si Colab se desconecta a la mitad, al volver a ejecutar se saltan los modelos ya terminados. Lo guardado lo genera **este mismo notebook** y cada archivo lleva una huella de los datos y parámetros, así que la reproducibilidad se mantiene. Para una ejecución limpia de entrega, ponlo en `False` o borra la carpeta `cache`.


In [1]:
%pip install -q lightgbm catboost

In [2]:
import os
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

SEED = 42
N_FOLDS = 5
USE_GPU = False      # True solo si activaste GPU en Colab
USE_CACHE = True

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

In [3]:
# Conectar Google Drive (solo en Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    current_dir = Path("/content/drive/MyDrive/Competition")   # <- cambia si tu carpeta está en otro lugar
except ImportError:
    current_dir = Path(os.environ.get("COMP_DIR", os.getcwd()))

input_dir = current_dir / "input"
cache_dir = current_dir / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)

train_file = input_dir / "train.csv"
test_file = input_dir / "test.csv"
sample_sub_file = input_dir / "sample_submission.csv"

assert train_file.exists() and test_file.exists() and sample_sub_file.exists(), \
    "Faltan archivos en input/. Revisa la ruta de current_dir."
print("Carpeta de trabajo:", current_dir)

Mounted at /content/drive
Carpeta de trabajo: /content/drive/MyDrive/Competition


## 2. Carga y validación de datos

Los `assert` detienen el notebook si algo no cuadra, por ejemplo si los IDs de test no coinciden con los del `sample_submission`. Es mejor que falle aquí y no que subas un CSV desalineado, que te daría un AUC cercano a 0.5.

In [4]:
train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
sample_sub = pd.read_csv(sample_sub_file)

y = train["TARGET"].to_numpy()

assert set(np.unique(y)) == {0, 1}
assert train["SK_ID_CURR"].is_unique and test["SK_ID_CURR"].is_unique
assert set(train["SK_ID_CURR"]).isdisjoint(set(test["SK_ID_CURR"]))
assert sample_sub["SK_ID_CURR"].equals(test["SK_ID_CURR"])

print(f"train: {train.shape} | test: {test.shape}")
print(f"Proporción de morosos: {y.mean():.4f}")

train: (171202, 34) | test: (61500, 33)
Proporción de morosos: 0.0807


## 3. Creación de variables

### 3.1 Variables base
Se agrupan en cinco familias:
- **Limpieza:** `DAYS_EMPLOYED = 365243` es un código de "no aplica" (casi siempre pensionados) y se convierte en faltante. Los días se pasan a años positivos.
- **Capacidad de pago:** crédito/ingreso, cuota/ingreso, crédito/precio del bien, ingreso por persona, etc. Es lo mismo que mira un analista de crédito.
- **Puntajes externos (`EXT_SOURCE_1/2/3`):** son las variables más predictivas. Se combinan con media, mínimo, máximo, producto y diferencias.
- **Tiempos relativos:** antigüedad laboral, de registro o de documento en proporción a la edad.
- **Faltantes:** cuántos datos faltan y en qué columnas. El hecho de que falte un dato puede ser informativo.

In [5]:
CAT_COLS = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE",
]


def base_features(raw):
    df = raw.drop(columns=["TARGET", "SK_ID_CURR"], errors="ignore").copy()

    # Limpieza
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    df["DAYS_BIRTH"] = df["DAYS_BIRTH"] / -365          # edad en años
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"] / -365    # años trabajando

    def ratio(name, num, den):
        df[name] = df[num] / df[den].replace(0, np.nan)

    # Capacidad de pago
    ratio("CREDIT_INCOME_PERC", "AMT_CREDIT", "AMT_INCOME_TOTAL")
    ratio("ANNUITY_INCOME_PERC", "AMT_ANNUITY", "AMT_INCOME_TOTAL")
    ratio("CREDIT_TERM", "AMT_ANNUITY", "AMT_CREDIT")
    ratio("DAYS_EMPLOYED_PERC", "DAYS_EMPLOYED", "DAYS_BIRTH")

    # Puntajes externos
    ext23 = df[["EXT_SOURCE_2", "EXT_SOURCE_3"]]
    df["EXT_SOURCES_PROD"] = df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"]
    for op in ["mean", "min", "max", "var"]:
        df[f"EXT_SOURCES_{op.upper()}"] = getattr(ext23, op)(axis=1)

    ext_all = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    for op in ["mean", "min", "max", "std"]:
        df[f"EXT_ALL_{op}"] = getattr(ext_all, op)(axis=1)
    df["EXT_ALL_missing"] = ext_all.isna().sum(axis=1)
    df["EXT_ALL_product"] = ext_all.prod(axis=1, min_count=3)
    for i, j in [(1, 2), (1, 3), (2, 3)]:
        df[f"EXT_{i}_{j}_product"] = df[f"EXT_SOURCE_{i}"] * df[f"EXT_SOURCE_{j}"]
        df[f"EXT_{i}_{j}_diff"] = df[f"EXT_SOURCE_{i}"] - df[f"EXT_SOURCE_{j}"]

    # Más razones financieras y familiares
    for name, num, den in [
        ("CREDIT_GOODS", "AMT_CREDIT", "AMT_GOODS_PRICE"),
        ("GOODS_INCOME", "AMT_GOODS_PRICE", "AMT_INCOME_TOTAL"),
        ("ANNUITY_GOODS", "AMT_ANNUITY", "AMT_GOODS_PRICE"),
        ("INCOME_PERSON", "AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS"),
        ("CREDIT_PERSON", "AMT_CREDIT", "CNT_FAM_MEMBERS"),
        ("ANNUITY_PERSON", "AMT_ANNUITY", "CNT_FAM_MEMBERS"),
        ("CHILDREN_SHARE", "CNT_CHILDREN", "CNT_FAM_MEMBERS"),
        ("CREDIT_ANNUITY", "AMT_CREDIT", "AMT_ANNUITY"),
    ]:
        ratio(name, num, den)
    df["CREDIT_GOODS_DIFF"] = df["AMT_CREDIT"] - df["AMT_GOODS_PRICE"]
    df["INCOME_AFTER_ANNUITY"] = df["AMT_INCOME_TOTAL"] - df["AMT_ANNUITY"]
    df["NON_CHILD_MEMBERS"] = df["CNT_FAM_MEMBERS"] - df["CNT_CHILDREN"]

    # Tiempos relativos
    df["EMPLOYED_MISSING"] = df["DAYS_EMPLOYED"].isna().astype(int)
    df["AGE_STARTED_WORK"] = df["DAYS_BIRTH"] - df["DAYS_EMPLOYED"]
    for c in ["DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE"]:
        df[f"{c}_AGE_RATIO"] = (-df[c] / 365) / df["DAYS_BIRTH"]

    # Faltantes
    df["MISSING_COUNT"] = raw.drop(columns=["TARGET", "SK_ID_CURR"], errors="ignore").isna().sum(axis=1)
    for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "OWN_CAR_AGE"]:
        df[f"{c}_MISSING"] = df[c].isna().astype(int)

    return df

### 3.2 Tasa de interés implícita

**Idea:** un préstamo con cuota fija sigue la fórmula de amortización

$$\frac{\text{Crédito}}{\text{Cuota}} = \frac{1-(1+r)^{-n}}{r}$$

donde $r$ es la tasa mensual y $n$ el número de pagos. Tenemos el crédito y la cuota, pero no $n$ ni $r$. Sin embargo:
- Con interés, el número de pagos siempre es **mayor** que Crédito/Cuota.
- Los plazos comerciales son estándar (6, 10, 12, 18, 24, 30, 36… meses).

Entonces se supone que $n$ es el plazo estándar más corto que sea ≥ Crédito/Cuota, y se despeja $r$ por bisección (un método numérico que va partiendo el intervalo a la mitad hasta encontrar la raíz).

**Por qué ayuda:** el prestamista asigna tasas más altas a los clientes que considera más riesgosos, así que la tasa contiene **su propia evaluación de riesgo**. Además se calcula la tasa que tendría el préstamo con cada plazo fijo (12, 18, 24, 30, 36 y 48 meses), para que el modelo elija la que más le sirva.

**Resultado medido:** en LightGBM, el AUC OOF pasó de 0.7614 a **0.7641** y mejoró en los 5 folds.

El plazo real es desconocido; esto es una aproximación. Para los créditos revolventes (tarjetas), la cuota es un porcentaje fijo y la "tasa" no tiene interpretación financiera, pero el modelo sabe distinguirlos con `NAME_CONTRACT_TYPE`.

In [6]:
STANDARD_TERMS = np.array([6, 10, 12, 18, 24, 30, 36, 42, 48, 54, 60])


def solve_monthly_rate(annuity_factor, n_payments, iters=60):
    """Bisección: encuentra r tal que (1-(1+r)^-n)/r = annuity_factor."""
    lo = np.full(len(annuity_factor), 1e-7)
    hi = np.full(len(annuity_factor), 0.3)
    for _ in range(iters):
        mid = (lo + hi) / 2
        f = (1 - (1 + mid) ** -n_payments) / mid   # decrece cuando r aumenta
        too_high_rate = f < annuity_factor
        hi = np.where(too_high_rate, mid, hi)
        lo = np.where(too_high_rate, lo, mid)
    return (lo + hi) / 2


def interest_features(df):
    factor = (df["AMT_CREDIT"] / df["AMT_ANNUITY"]).to_numpy()
    out = pd.DataFrame(index=df.index)

    # Plazo estándar más corto que sea >= crédito/cuota
    n = np.full(len(factor), np.nan)
    for t in STANDARD_TERMS[::-1]:
        n = np.where(factor <= t, t, n)
    rate = solve_monthly_rate(factor, n)
    rate[np.isnan(n) | np.isnan(factor)] = np.nan

    out["IMPL_TERM"] = n
    out["IMPL_RATE"] = rate
    out["TERM_GAP"] = n - factor

    # Tasa que tendría el préstamo con cada plazo fijo
    for t in [12, 18, 24, 30, 36, 48]:
        r_t = solve_monthly_rate(factor, np.full(len(factor), float(t)))
        r_t[~(factor < t)] = np.nan          # plazo imposible: no aplica
        out[f"RATE_N{t}"] = r_t
    return out


def create_features(raw):
    df = base_features(raw)
    df = pd.concat([df, interest_features(df)], axis=1)
    return df.replace([np.inf, -np.inf], np.nan)


X = create_features(train)
X_test = create_features(test)
assert X.columns.equals(X_test.columns)

# Categorías: el mismo diccionario para train y test (faltante = "__MISSING__")
for c in CAT_COLS:
    values = pd.concat([X[c], X_test[c]]).astype("object").fillna("__MISSING__")
    dtype = pd.CategoricalDtype(sorted(values.unique()))
    X[c] = X[c].astype("object").fillna("__MISSING__").astype(dtype)
    X_test[c] = X_test[c].astype("object").fillna("__MISSING__").astype(dtype)

print("Número de variables:", X.shape[1])
X[["IMPL_TERM", "IMPL_RATE", "TERM_GAP"]].describe().T

Número de variables: 83


,count,mean,std,min,25%,50%,75%,max
IMPL_TERM,171202.0,24.759243,8.289063,1.000000e+01,18.000000,24.000000,30.000000,48.000000
IMPL_RATE,171202.0,0.013333,0.009840,1.000000e-07,0.005195,0.011829,0.017361,0.046162
TERM_GAP,171202.0,3.154003,1.689542,0.000000e+00,1.797315,3.323147,4.496275,5.998611


### 3.3 Clientes repetidos

Algunos clientes pidieron **más de un préstamo**, y sus solicitudes están repartidas entre train y test. El dataset no trae un ID de persona, pero se puede reconstruir:

- Los `DAYS_*` se cuentan **desde la fecha de cada solicitud**. Si la misma persona pide dos préstamos con 100 días de diferencia, todos sus `DAYS_*` cambian en 100.
- Pero la **diferencia** entre dos de ellos no cambia. Por ejemplo, `DAYS_BIRTH − DAYS_ID_PUBLISH` es la edad que tenía al emitir su documento, y es la misma en todas sus solicitudes.

Se usan dos "huellas" del cliente:
- **K1** = sexo + (nacimiento − emisión del documento) + (nacimiento − registro)
- **K2** = sexo + (nacimiento − registro) + (nacimiento − inicio de empleo)

Con eso se crean dos tipos de variables:
1. **Sin TARGET** (se calculan aquí): cuántas solicitudes tiene el cliente, cuál es esta en orden, cuántos días pasaron entre la primera y la última, y cómo se compara su ingreso o crédito con el de sus otras solicitudes.
2. **Con TARGET** (se calculan **dentro de cada fold**, sección 5): la tasa de mora de **las otras** solicitudes del mismo cliente, usando solo etiquetas del fold de entrenamiento. Nunca se usa la etiqueta de la propia fila ni la del fold de validación, así que no hay fuga.

**Evidencia:** si otro préstamo del mismo cliente cayó en mora, la probabilidad de mora es 25%, contra 4.8% cuando el otro préstamo se pagó (la tasa base es 8%).

**Reglas:** usa solo `train.csv` y `test.csv` y todas las predicciones salen del modelo, así que no hay datos externos ni hand-labeling.

In [7]:
def client_keys(df):
    gender = df["CODE_GENDER"].astype(str)
    birth = df["DAYS_BIRTH"]
    id_age = (birth - df["DAYS_ID_PUBLISH"]).astype(str)
    reg_age = (birth - df["DAYS_REGISTRATION"]).round(0).astype(str)
    emp = pd.Series(np.where(df["DAYS_EMPLOYED"] == 365243, np.nan, birth - df["DAYS_EMPLOYED"]))
    k1 = gender + "|" + id_age + "|" + reg_age
    k2 = (gender + "|" + reg_age + "|" + emp.astype(str)).where(emp.notna(), None)   # sin empleo: no aplica
    return {"K1": k1, "K2": k2}


all_raw = pd.concat([train.drop(columns="TARGET"), test], ignore_index=True)
CLIENT_KEYS = client_keys(all_raw)
N_TRAIN = len(train)

static = pd.DataFrame(index=all_raw.index)
for name, key in CLIENT_KEYS.items():
    g = all_raw.groupby(key, dropna=True)
    static[f"{name}_N_APPS"] = g["SK_ID_CURR"].transform("size")
    static[f"{name}_APP_ORDER"] = g["DAYS_BIRTH"].rank(ascending=False)      # 1 = primera solicitud
    static[f"{name}_DAYS_SPAN"] = g["DAYS_BIRTH"].transform("max") - g["DAYS_BIRTH"].transform("min")
    static[f"{name}_INC_VS_GRP"] = all_raw["AMT_INCOME_TOTAL"] / g["AMT_INCOME_TOTAL"].transform("mean")
    static[f"{name}_CRED_VS_GRP"] = all_raw["AMT_CREDIT"] / g["AMT_CREDIT"].transform("mean")

X = pd.concat([X.reset_index(drop=True), static.iloc[:N_TRAIN].reset_index(drop=True)], axis=1)
X_test = pd.concat([X_test.reset_index(drop=True), static.iloc[N_TRAIN:].reset_index(drop=True)], axis=1)

y_all = np.concatenate([y, np.zeros(len(test))])


def client_target_features(labeled_idx):
    """Tasa de mora de las OTRAS solicitudes del cliente, usando solo las etiquetas de labeled_idx."""
    labeled = np.zeros(len(y_all), dtype=bool)
    labeled[labeled_idx] = True
    cols = {}
    for name, key in CLIENT_KEYS.items():
        df = pd.DataFrame({"k": key, "y": np.where(labeled, y_all, 0.0), "c": labeled.astype(float)})
        g = df.groupby("k", dropna=True)
        other_sum = g["y"].transform("sum") - df["y"]      # se resta la propia fila
        other_cnt = g["c"].transform("sum") - df["c"]
        cols[f"{name}_OTHER_TGT"] = np.where(other_cnt > 0, other_sum / other_cnt.replace(0, np.nan), np.nan)
        cols[f"{name}_OTHER_CNT"] = other_cnt.fillna(0).to_numpy()
    F = pd.DataFrame(cols)
    return F.iloc[:N_TRAIN].reset_index(drop=True), F.iloc[N_TRAIN:].reset_index(drop=True)


repeated = (static["K1_N_APPS"] > 1) | (static["K2_N_APPS"] > 1)
print(f"Solicitudes de clientes repetidos: train {repeated[:N_TRAIN].sum():,} | test {repeated[N_TRAIN:].sum():,}")

Solicitudes de clientes repetidos: train 7,425 | test 2,544


### 3.4 Vista B: imputación de los EXT_SOURCE

Se predice cada EXT_SOURCE faltante a partir del resto del perfil, sin usar TARGET. Para los clientes que sí lo tienen, la predicción es OOF. La vista A es `X` y la vista B es `X` + estas variables.

In [8]:
from sklearn.model_selection import KFold

EXT_COLS = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]


def ext_imputation_features(X, X_test, seed=SEED):
    full = pd.concat([X, X_test], ignore_index=True)
    base_cols = [c for c in full.columns
                 if not (c.startswith("EXT") or c.endswith("_MISSING") or c == "MISSING_COUNT"
                         or c.startswith("K1_") or c.startswith("K2_"))]
    params = dict(objective="regression", learning_rate=0.08, num_leaves=63,
                  min_child_samples=50, colsample_bytree=0.7, subsample=0.8, subsample_freq=1,
                  n_estimators=600, verbose=-1, n_jobs=-1, random_state=seed)
    out = {}
    for col in EXT_COLS:
        predictors = base_cols + [e for e in EXT_COLS if e != col]
        observed = full[col].notna().to_numpy()
        obs_idx = np.where(observed)[0]
        pred = np.zeros(len(full))
        missing_pred = np.zeros((~observed).sum())
        for tr, va in KFold(n_splits=5, shuffle=True, random_state=seed).split(obs_idx):
            reg = lgb.LGBMRegressor(**params)
            reg.fit(full.iloc[obs_idx[tr]][predictors], full[col].to_numpy()[obs_idx[tr]])
            pred[obs_idx[va]] = reg.predict(full.iloc[obs_idx[va]][predictors])
            if (~observed).any():
                missing_pred += reg.predict(full.loc[~observed, predictors]) / 5
        pred[~observed] = missing_pred
        real = full[col].to_numpy()
        out[f"{col}_PRED"] = pred
        out[f"{col}_FILLED"] = np.where(observed, real, pred)
        out[f"{col}_RESID"] = np.where(observed, real - pred, np.nan)
    new = pd.DataFrame(out)
    filled = new[[f"{c}_FILLED" for c in EXT_COLS]]
    new["EXTF_mean"] = filled.mean(axis=1)
    new["EXTF_min"] = filled.min(axis=1)
    new["EXTF_max"] = filled.max(axis=1)
    new["EXTF_prod"] = filled.prod(axis=1)
    new["EXTF_w"] = 2 * filled.iloc[:, 0] + filled.iloc[:, 1] + 3 * filled.iloc[:, 2]
    return new.iloc[:len(X)].reset_index(drop=True), new.iloc[len(X):].reset_index(drop=True)


imp_path = cache_dir / "ext_imputation.pkl"
if USE_CACHE and imp_path.exists():
    imp_train, imp_test = pd.read_pickle(imp_path)
    print("[cache] imputación cargada")
else:
    imp_train, imp_test = ext_imputation_features(X, X_test)
    pd.to_pickle((imp_train, imp_test), imp_path)

X_imp = pd.concat([X, imp_train], axis=1)
X_test_imp = pd.concat([X_test, imp_test], axis=1)

# Copias con categorías como texto para CatBoost
def as_text_cats(df):
    df = df.copy()
    for c in CAT_COLS:
        df[c] = df[c].astype(str)
    return df

VIEWS = {"A": (X, X_test), "B": (X_imp, X_test_imp)}
VIEWS_CB = {v: (as_text_cats(a), as_text_cats(b)) for v, (a, b) in VIEWS.items()}
print(f"Vista A: {X.shape[1]} variables (+4 por fold) | Vista B: {X_imp.shape[1]} variables (+4 por fold)")

Vista A: 93 variables (+4 por fold) | Vista B: 107 variables (+4 por fold)


## 4. Validación cruzada (esquema común)

**StratifiedKFold de 5 pliegues:** cada parte tiene la misma proporción de morosos (8%).
- **OOF (out-of-fold):** cada cliente de train recibe una predicción hecha por un modelo que *no lo vio*.
- **Predicción de test:** es el promedio de los 5 modelos del fold.



In [9]:
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X, y))


def report(name, oof):
    fold_aucs = [roc_auc_score(y[va], oof[va]) for _, va in folds]
    print(f"{name}: AUC OOF = {roc_auc_score(y, oof):.5f} | por fold = {np.round(fold_aucs, 4)}")


def cached(name, train_fn):
    """Carga un resultado guardado si existe; si no, lo calcula y lo guarda.
    El nombre del archivo incluye FINGERPRINT (huella de datos, variables y parámetros):
    si algo cambia, la huella cambia y no se reutilizan resultados viejos."""
    path = cache_dir / f"{name}_{FINGERPRINT}.npz"
    if USE_CACHE and path.exists():
        saved = np.load(path)
        print(f"[cache] {name} cargado")
        return saved["a"], saved["b"]
    a, b = train_fn()
    np.savez_compressed(path, a=a, b=b)
    return a, b


def to_rank(p):
    return rankdata(p) / len(p)

## 5. Paso 1: validación cruzada (mide el modelo y decide cuántos árboles usar)

Cada uno de los 4 modelos (LightGBM y CatBoost × vistas A y B) se entrena con 5 folds. Sirve para tres cosas:
1. **Medir** el AUC OOF de cada modelo de forma honesta.
2. **Decidir cuántos árboles** necesita cada modelo: el promedio de la mejor iteración de los 5 folds.
3. **Elegir los pesos** del ensamble (sección 7).

Usa una sola semilla (42): para medir y contar árboles es suficiente, y deja el tiempo de cómputo para el paso 2.

**LightGBM:** `num_leaves=12`, `min_child_samples=200` (árboles pequeños, hojas con muchos clientes) y `colsample_bytree=0.35`. Vista A con `learning_rate=0.03`; vista B con `0.01`.

**CatBoost:** categorías con *ordered target statistics* y árboles simétricos (`depth=6`).

**Dentro de cada fold** se agregan las 4 variables de mora de otras solicitudes del cliente (3.3), calculadas solo con las etiquetas del fold de entrenamiento.

In [10]:
LGB_PARAMS = dict(
    objective="binary",
    num_leaves=12, min_child_samples=200, colsample_bytree=0.35,
    subsample=0.85, subsample_freq=1, reg_alpha=0.5, reg_lambda=10,
    cat_smooth=50, cat_l2=10, max_cat_to_onehot=4,
    n_jobs=-1, verbose=-1,
)
LGB_LR = {"A": 0.03, "B": 0.01}
LGB_EARLY_STOP = {"A": 200, "B": 300}

CB_PARAMS = dict(
    learning_rate=0.05, depth=6, l2_leaf_reg=10, border_count=254, verbose=0,
    task_type="GPU" if USE_GPU else "CPU",
)
MODELS = [("lgb", "A"), ("lgb", "B"), ("cb", "A"), ("cb", "B")]

# Huella para la caché: datos + variables + parámetros + folds
import hashlib
_h = hashlib.md5()
for _part in [pd.util.hash_pandas_object(train).values.tobytes(), pd.util.hash_pandas_object(test).values.tobytes(),
              str(list(X_imp.columns)).encode(), str(sorted(LGB_PARAMS.items())).encode(), str(LGB_LR).encode(),
              str(LGB_EARLY_STOP).encode(), str(sorted(CB_PARAMS.items())).encode(), str((SEED, N_FOLDS)).encode()]:
    _h.update(_part)
FINGERPRINT = _h.hexdigest()[:10]
print("Huella de caché:", FINGERPRINT)


def cv_model(kind, view):
    """Validación cruzada de un modelo. Devuelve predicciones OOF y la mejor iteración de cada fold."""
    Xv, _ = VIEWS_CB[view] if kind == "cb" else VIEWS[view]
    oof = np.zeros(len(Xv))
    best_iters = []
    for k, (tr, va) in enumerate(folds):
        F_train, _ = client_target_features(tr)          # solo etiquetas del fold de entrenamiento
        Xa = pd.concat([Xv, F_train], axis=1)
        if kind == "lgb":
            model = lgb.LGBMClassifier(**LGB_PARAMS, n_estimators=15000, learning_rate=LGB_LR[view], random_state=SEED)
            model.fit(Xa.iloc[tr], y[tr], eval_set=[(Xa.iloc[va], y[va])], eval_metric="auc",
                      callbacks=[lgb.early_stopping(LGB_EARLY_STOP[view], verbose=False)])
            oof[va] = model.predict_proba(Xa.iloc[va])[:, 1]
            best_iters.append(model.best_iteration_)
        else:
            model = CatBoostClassifier(**CB_PARAMS, iterations=6000, eval_metric="AUC", od_type="Iter", od_wait=300,
                                       random_seed=SEED)
            model.fit(Pool(Xa.iloc[tr], y[tr], cat_features=CAT_COLS),
                      eval_set=Pool(Xa.iloc[va], y[va], cat_features=CAT_COLS), use_best_model=True)
            oof[va] = model.predict_proba(Xa.iloc[va])[:, 1]
            best_iters.append(model.get_best_iteration() + 1)
        print(f"  {kind}-{view} fold {k+1}: AUC {roc_auc_score(y[va], oof[va]):.5f} | iteraciones {best_iters[-1]}")
        del model; gc.collect()
    return oof, np.array(best_iters)


model_oofs, model_iters = {}, {}
for kind, view in MODELS:
    name = f"{kind}_{view}"
    model_oofs[name], model_iters[name] = cached(f"cv_{name}", lambda: cv_model(kind, view))
    report(name, model_oofs[name])
    print(f"  iteraciones por fold: {model_iters[name].astype(int).tolist()}")

Huella de caché: 507d700916
  lgb-A fold 1: AUC 0.76173 | iteraciones 979
  lgb-A fold 2: AUC 0.76695 | iteraciones 1524
  lgb-A fold 3: AUC 0.76991 | iteraciones 1218
  lgb-A fold 4: AUC 0.76332 | iteraciones 1224
  lgb-A fold 5: AUC 0.76701 | iteraciones 1018
lgb_A: AUC OOF = 0.76578 | por fold = [0.7617 0.767  0.7699 0.7633 0.767 ]
  iteraciones por fold: [979, 1524, 1218, 1224, 1018]
  lgb-B fold 1: AUC 0.76437 | iteraciones 2723
  lgb-B fold 2: AUC 0.76770 | iteraciones 4503
  lgb-B fold 3: AUC 0.77161 | iteraciones 3068
  lgb-B fold 4: AUC 0.76542 | iteraciones 3509
  lgb-B fold 5: AUC 0.76961 | iteraciones 3248
lgb_B: AUC OOF = 0.76772 | por fold = [0.7644 0.7677 0.7716 0.7654 0.7696]
  iteraciones por fold: [2723, 4503, 3068, 3509, 3248]
  cb-A fold 1: AUC 0.76318 | iteraciones 1656
  cb-A fold 2: AUC 0.76725 | iteraciones 1546
  cb-A fold 3: AUC 0.76889 | iteraciones 1370
  cb-A fold 4: AUC 0.76327 | iteraciones 984
  cb-A fold 5: AUC 0.76743 | iteraciones 1704
cb_A: AUC OOF =

## 6. Paso 2: reentrenar con el 100% de los datos

La forma habitual de predecir el test es promediar los 5 modelos de los folds, pero cada uno de ellos vio solo el **80%** de los datos. Aquí, en cambio, cada modelo se vuelve a entrenar con **todo** el train y ese es el que predice el test.

**Por qué ayuda:**
1. **25% más datos** para aprender.
2. **Más ejemplos de clientes repetidos.** La variable "mora de las otras solicitudes" solo existe cuando la otra solicitud tiene etiqueta. Con el 100% de las etiquetas, el modelo ve muchos más pares completos durante el entrenamiento, y en el test más clientes tienen una solicitud etiquetada de referencia.

**Sin fuga de información:** para cada fila de train, la variable usa las etiquetas de las **otras** solicitudes del cliente, nunca la propia (*leave-one-out*). Para el test se usan todas las etiquetas de train. El test nunca aporta etiquetas porque no las tiene.

**¿Cuántos árboles?** Sin fold de validación no hay parada temprana. Se usa el promedio de iteraciones del paso 1 multiplicado por `REFIT_MULT = 1.25`, porque con más datos el modelo aguanta más árboles.

**Cómo se validó** (el AUC OOF no puede medir esto, porque el OOF siempre viene de modelos entrenados con el 80%): se hizo una **simulación anidada**. Un fold se apartó como si fuera el test, sin usar sus etiquetas, y se compararon los dos procedimientos sobre el 80% restante:

| AUC en el fold apartado | Promedio de 5 modelos de fold | Reentrenado con todo |
|---|---|---|
| LightGBM-B, media de 5 folds apartados | 0.76608 | **0.76693** (+0.00086, mejora en 5 de 5) |
| CatBoost-B, media de 2 folds apartados | 0.76326 | **0.76443** (+0.00117, mejora en 2 de 2) |

Cada modelo se reentrena con 3 semillas (42, 7, 2026) y se promedian los rangos.


In [11]:
REFIT_MULT = 1.25
REFIT_SEEDS = [42, 7, 2026]

F_full_train, F_full_test = client_target_features(np.arange(N_TRAIN))    # todas las etiquetas de train


def refit_model(kind, view, seed, n_iter):
    Xv, Xtv = VIEWS_CB[view] if kind == "cb" else VIEWS[view]
    Xa = pd.concat([Xv, F_full_train], axis=1)
    Xta = pd.concat([Xtv, F_full_test], axis=1)
    if kind == "lgb":
        model = lgb.LGBMClassifier(**LGB_PARAMS, n_estimators=n_iter, learning_rate=LGB_LR[view], random_state=seed)
        model.fit(Xa, y)
        test_pred = model.predict_proba(Xta)[:, 1]
    else:
        model = CatBoostClassifier(**CB_PARAMS, iterations=n_iter, random_seed=seed)
        model.fit(Pool(Xa, y, cat_features=CAT_COLS))
        test_pred = model.predict_proba(Pool(Xta, cat_features=CAT_COLS))[:, 1]
    del model; gc.collect()
    return test_pred, np.array([n_iter])


model_tests = {}
for kind, view in MODELS:
    name = f"{kind}_{view}"
    n_iter = int(round(model_iters[name].mean() * REFIT_MULT))
    preds = []
    for s in REFIT_SEEDS:
        p, _ = cached(f"refit_{name}_s{s}_n{n_iter}", lambda s=s: refit_model(kind, view, s, n_iter))
        preds.append(to_rank(p))
    model_tests[name] = np.mean(preds, axis=0)
    print(f"{name}: reentrenado con {n_iter} iteraciones × {len(REFIT_SEEDS)} semillas")

lgb_A: reentrenado con 1491 iteraciones × 3 semillas
lgb_B: reentrenado con 4263 iteraciones × 3 semillas
cb_A: reentrenado con 1815 iteraciones × 3 semillas
cb_B: reentrenado con 1692 iteraciones × 3 semillas


## 7. Ensamble de 4 modelos

Se promedian **rangos**: el AUC solo depende del orden, y así ningún modelo domina por su escala. Los pesos se buscan con pasos de 0.05 sobre el AUC OOF del paso 1 (1,771 combinaciones), **no con el leaderboard**, y se aplican a las predicciones de test de los modelos reentrenados.

El AUC OOF de esta sección mide los modelos entrenados con el 80% de los datos. No refleja la ganancia del reentrenamiento con el 100%, que está medida en la simulación de la sección 6.


In [12]:
from itertools import product

oofs = {k: to_rank(v) for k, v in model_oofs.items()}
tests = {k: to_rank(v) for k, v in model_tests.items()}
names = list(oofs)

steps = np.round(np.arange(0, 1.0001, 0.05), 2)
best_weights, best_auc = None, -1
for w in product(steps, repeat=len(names) - 1):
    last = round(1 - sum(w), 2)
    if last < 0:
        continue
    ws = dict(zip(names, [float(v) for v in w] + [float(abs(last))]))
    auc = roc_auc_score(y, sum(ws[m] * oofs[m] for m in names))
    if auc > best_auc:
        best_weights, best_auc = ws, auc

weights = best_weights
print("Pesos elegidos:", weights)
for m in names:
    print(f"  {m:6s} AUC OOF {roc_auc_score(y, oofs[m]):.5f}")
print(f"  ENSAMBLE AUC OOF {best_auc:.5f}")
report("Ensamble", sum(weights[m] * oofs[m] for m in names))

Pesos elegidos: {'lgb_A': 0.1, 'lgb_B': 0.4, 'cb_A': 0.25, 'cb_B': 0.25}
  lgb_A  AUC OOF 0.76578
  lgb_B  AUC OOF 0.76772
  cb_A   AUC OOF 0.76595
  cb_B   AUC OOF 0.76710
  ENSAMBLE AUC OOF 0.76948
Ensamble: AUC OOF = 0.76948 | por fold = [0.7664 0.7702 0.7731 0.7671 0.7708]


## 8. Predicción final



In [13]:
pred = sum(weights[m] * tests[m] for m in weights)

assert pred.shape == (len(sample_sub),)
assert np.isfinite(pred).all()
assert ((pred >= 0) & (pred <= 1)).all()

### 8.1 Saving the prediction as CSV file [DO NOT CHANGE]
**WARNING**: DO **NOT** CHANGE THE CODES BELOW!!!

In [14]:
# Put the prediction into the format of submission
sample_sub['TARGET'] = pred
sample_sub

,SK_ID_CURR,TARGET
0,171202,0.347513
1,171203,0.931685
2,171204,0.841277
3,171205,0.724111
4,171206,0.881441
...,...,...
61495,232697,0.840829
61496,232698,0.127133
61497,232699,0.488944
61498,232700,0.941925


In [15]:
# Create the "output" directory if it doesn't exist
output_dir = current_dir / "output"
os.makedirs(output_dir, exist_ok=True)

# Specify the new output file path
output_file = output_dir / "submission.csv"

# Save the CSV file to the "output" directory
sample_sub.to_csv(output_file, index=False)